In [67]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Define state
class MyState(TypedDict):
    length: float
    breadth: float
    area: float

# Create graph
graph = StateGraph(MyState)

# Node function
def calculate_area(state: MyState) -> dict:
    length = state["length"]
    breadth = state["breadth"]

    

    return {"area": length * breadth}

# Add node
graph.add_node("area_node", calculate_area)

# Define flow
graph.add_edge(START, "area_node")
graph.add_edge("area_node", END)

# Compile graph
app = graph.compile()

# Run
result = app.invoke({"length": 45, "breadth": 34})

print(result)

{'length': 45, 'breadth': 34, 'area': 1530}


In [68]:
from typing import TypedDict,Literal
from langgraph.graph import StateGraph,START,END

class chatstate(TypedDict):
    user_msg:str
    category:str
    response:str

def route_by_category(state: chatstate) -> Literal["technical", "billing", "general"]:
    """
    look at category in state and decide where to go.
    this function only makes routing decision...it doesn't update state.
    """
    return state.get("category", "general")


def classify_node(state: chatstate) -> dict:
    msg = state["user_msg"].lower()
    if any(w in msg for w in ["error", "bug", "crash", "api", "technical"]):
        return {"category": "technical"}
    elif any(w in msg for w in ["bill", "payment", "invoice", "refund"]):
        return {"category": "billing"}
    else:
        return {"category": "general"}
    

def technical_node(state:chatstate)->dict:
    return {"response": f"[Technical] '{state['user_msg']}' — Tech team will investigate."}

def billing_node(state:chatstate)->dict:
    return {"response": f"[billing] '{state['user_msg']}' — Please share account ID."}
def general_node(state:chatstate)->dict:
    return {"response": f"[General] '{state['user_msg']}' — Happy to help."}
    
graph=StateGraph(chatstate)

graph.add_node("classify",classify_node)
graph.add_node("technical",technical_node)
graph.add_node("billing",billing_node)
graph.add_node("general",general_node)


graph.add_edge(START,"classify")

graph.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "technical":"technical",
        "billing":"billing",
        "general":"general",

    }
    )


graph.add_edge("technical",END)
graph.add_edge("billing",END)
graph.add_edge("general",END)


graph = graph.compile()
result = graph.invoke({"user_msg": "where is my refund"})
print(result["response"])


[billing] 'where is my refund' — Please share account ID.
